# 4 · The training surface

`harness/train.py` and `harness/model.py` are **agent-modifiable**. This chapter
tours the knobs the agent turns: model capacity, the optimizer/schedule,
augmentation, and the **compute budget** that makes every change comparable.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))   # make `harness` importable
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt

from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    print("building a small bdd-tiny ...")
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA)

dataset: ../data/bdd-tiny.lance


### Capacity vs. budget — bigger isn't free

In [2]:
import time
from harness.config import load_config
from harness.train import train
from harness.evaluator import evaluate

rows = []
for width in [8, 16, 32]:
    cfg = load_config("base", {"model": {"width": width}, "budget": {"max_epochs": 12}})
    t = time.time(); m = train(cfg, DATA); dt = time.time() - t
    r = evaluate(m, DATA)
    rows.append((width, r["score"], r["worst_group"], r["overall_accuracy"], round(dt, 1)))
    print(f"width={width:2d}  score={r['score']:.3f}  fog={r['per_weather']['foggy']:.3f}  overall={r['overall_accuracy']:.3f}  {dt:.1f}s")

width= 8  score=0.838  fog=0.600  overall=0.951  4.3s


width=16  score=0.883  fog=0.644  overall=0.956  3.9s


width=32  score=0.952  fog=0.711  overall=0.962  7.7s


### The budget caps training — a change that needs more compute isn't valid

In [3]:
for me in [2, 6, 12]:
    cfg = load_config("base", {"budget": {"max_epochs": me, "max_seconds": 60}})
    m = train(cfg, DATA)
    print(f"max_epochs={me:2d} -> ran {m.metrics['epochs_run']} epochs, val={m.metrics['val_acc']:.3f}, {m.metrics['seconds']:.1f}s")

max_epochs= 2 -> ran 2 epochs, val=0.922, 0.6s


max_epochs= 6 -> ran 6 epochs, val=0.971, 2.1s


max_epochs=12 -> ran 12 epochs, val=0.973, 4.0s


### Augmentation knobs (generic robustness)

In [4]:
for aug in [{}, {"hflip": True}, {"brightness_jitter": 0.3}]:
    cfg = load_config("base", {"augment": aug})
    r = evaluate(train(cfg, DATA), DATA)
    print(f"aug={str(aug):28s} score={r['score']:.3f} fog={r['per_weather']['foggy']:.3f}")

aug={}                           score=0.883 fog=0.644


aug={'hflip': True}              score=0.906 fog=0.667


aug={'brightness_jitter': 0.3}   score=0.972 fog=0.733


Capacity and epochs help a little, but notice the **foggy** slice barely moves —
generic knobs don't fix a rare, degraded slice. That takes targeted
**hard-example mining**, which is Chapter 5.